In [2]:
import matplotlib.pyplot as plt
import numpy as np
import arc
import pairinteraction as pi
if pi.Database.get_global_database() is None:
    pi.Database.initialize_global_database(download_missing=False)
import os
from IPython.display import Latex
import scipy
import pandas as pd
from pandas import DataFrame
h = scipy.constants.h
epsilon_0 = scipy.constants.epsilon_0
a_0 = scipy.constants.physical_constants['Bohr radius']
e = scipy.constants.e


In [3]:
import os
if pi.Database.get_global_database() is None:
    pi.Database.initialize_global_database(download_missing=False)
atom1, atom2 = 'Rb', 'Cs'
colours = ('orange', 'green', 'red', 'blue')
orbs = ('s', 'p', 'd', 'f')
difs = np.arange(-8,9)
changes = np.arange(-3, 4)
energy_unit = 'GHz'

min_n = 30
max_n = 120
ns = np.arange(min_n,max_n)

js = [[1/2], [1/2, 3/2], [3/2, 5/2], [5/2, 7/2]]

jss = [1/2]
jps = [1/2, 3/2]
jds = [3/2, 5/2]
jfs = [5/2, 7/2]

l1, l2 = 1, 1
j1, j2 = js[l1][1], js[l2][1]
l1_1, l2_1 = 2, 0

jp_deltas = []
channels = []
preferQuantumDefects = True
# C3k coefficients
c3ks = []
df = {'n1_0' : [], 'l1_0' : [], 'j1_0' : [],
      'n2_0' : [], 'l2_0' : [], 'j2_0' : [],
      'n1_1' : [], 'l1_1' : [], 'j1_1' : [],
      'n2_1' : [], 'l2_1' : [], 'j2_1' : [],
      'defect' : [], 'c3k' : []}
w=1

targets = []
resonants = []
min_defects = []
c_3ks = []
basis = pi.BasisAtom(atom1, n = (min_n-difs[-1]-changes[-1]-1,max_n+difs[-1]+changes[-1]+1), l = (0,6))

num=1
for l1_1 in [l1_1]:
    for l2_1 in [l2_1]:
        for dif in difs:
            for change1 in changes:
                fig, axs = plt.subplots(nrows = len(changes), sharex = True, figsize=(5,20))
                
                for change2 in changes:
                    jpdelta = []

                    labels = []
            
                    for j1_1 in js[l1_1]:
                    
                        for j2_1 in js[l2_1]:

                            delta = []


                            for n in ns:
                                nb = n+dif
                                
                                # Rb lowering n
                                ket1 = pi.KetAtom(atom1, n=n, l=l1, j=j1, m=j1, s=1/2)
                                ket2 = pi.KetAtom(atom2, n=nb, l=l2, j=j2, m=j2, s=1/2)
                                ket1_1 = pi.KetAtom(atom1, n=n+change1, l=l1_1, j=j1_1, m=j1_1, s=1/2)
                                ket2_1 = pi.KetAtom(atom2, n=nb+change2, l=l2_1, j=j2_1, m=j2_1, s=1/2)
                                
                                defect_1 = ket1_1.get_energy(energy_unit) - ket1.get_energy(energy_unit)
                                defect_2 = ket2_1.get_energy(energy_unit) - ket2.get_energy(energy_unit)
                                defect = defect_1 + defect_2
                                delta.append(defect)
                            
                            jpdelta.append(delta)
                            labels.append(str(j1_1) + ', ' + str(j2_1))
                            
                            min_defect = min(np.array(delta), key=abs)

                            for i in [-1,0,1]:
                                
                                
                                try:
                                    assoc_n = ns[delta.index(min_defect) + i]
                                    defect = delta[delta.index(min_defect) + i]
                                    ket1 = pi.KetAtom(atom1, n=assoc_n, l=l1, j=j1, m=j1, s=1/2)
                                    ket2 = pi.KetAtom(atom1, n=assoc_n+dif, l=l2, j=j2, m=j2, s=1/2)
                                    ket1_1 = pi.KetAtom(atom1, n=assoc_n+change1, l=l1_1, j=j1_1, m=j1_1, s=1/2)
                                    ket2_1 = pi.KetAtom(atom1, n=assoc_n+dif-change2, l=l2_1, j=j2_1, m=j2_1, s=1/2)
                                
                                    c = {"n1_0" : assoc_n, 'l1_0' : l1, 'j1_0' : j1, 
                                        'n2_0' : assoc_n+dif, 'l2_0' : l2, 'j2_0' : j2, 
                                        'n1_1' : assoc_n+change1, 'l1_1' : l1_1, 'j1_1' : j1_1, 
                                        'n2_1' : assoc_n+dif-change2, 'l2_1' : l2_1, 'j2_1' : j2_1, 
                                        'defect' : defect}
                                    pairi_c = [pi.StateAtom(ket1, basis), pi.StateAtom(ket2, basis), pi.StateAtom(ket1_1, basis), pi.StateAtom(ket2_1, basis)]                                                             
                                    one = True
                            
                                except  IndexError:
                                    continue
                                        
                                        
                                if atom1 == 'Rb':
                                    rb_matrix_element_j = arc.Rubidium().getReducedMatrixElementJ(c['n1_0'], l1, j1, c['n1_1'], l1_1, j1_1)     
                                else:
                                    rb_matrix_element_j = arc.Caesium().getReducedMatrixElementJ(c['n1_0'], l1, j1, c['n1_1'], l1_1, j1_1)
                                if atom2 == 'Rb':
                                    cs_matrix_element_j = arc.Rubidium().getReducedMatrixElementJ(c['n2_0'], l2, j2, c['n2_1'], l2_1, j2_1)     
                                else:
                                    cs_matrix_element_j = arc.Caesium().getReducedMatrixElementJ(c['n2_0'], l2, j2, c['n2_1'], l2_1, j2_1)
                                
                                c3k = (a_0[0]*e) **2 / h / (4*np.pi*epsilon_0) * 1e9 * cs_matrix_element_j * rb_matrix_element_j/np.sqrt(2*j1_1+1)/np.sqrt(2*j2_1+1)
                                
                                c.update({'c3k': c3k})
                                
                                level_spacing = pi.KetAtom(atom1, n=assoc_n, l=l1, j=j1, m=1/2).get_energy(unit = energy_unit) - pi.KetAtom(atom1, n=assoc_n+1, l=l1, j=j1, m=1/2).get_energy(unit = energy_unit)

                        
                                # print('defect ' + str(abs(defect)))
                                # print('level ' + str(level_spacing))
                                # print(abs(defect) - 5 * abs(level_spacing))
                                # print('c3k ' + str(c3k))
                                # print(c['n1_0'])
                                if abs(defect) < 0.005 * abs(level_spacing) and 40 < c['n1_0'] < 90 and abs(c3k) > 1:

                                    print(num)
                                    num+=1
                                    print(f'channel = {pairi_c[0]}, {pairi_c[1]} -> {pairi_c[2]}, {pairi_c[3]}')
                                    # print(f'channel {jps.index(jp)+1} = |Rb {c['n1_0']}s_1/2, Rb {c['n2_0']}s_1/2> -> |Rb {c['n1_1']}p_{c['j1_1']}, Rb {c['n2_1']}p_{c['j2_1']}>')
                                    print(f'min defect = {defect*1e3} MHz')
                                    print(f'c3k = {c3k}')
                                    s1 = ket1
                                    s2 = ket2
                                    s3 = ket1_1
                                    s4 = ket2_1

                                    target = fr'${s1.n}{orbs[int(s1.l)]}_{{{s1.j}}}$ ${s2.n}{orbs[int(s2.l)]}_{{{s2.j}}}$'
                                    resonant = fr'${s3.n}{orbs[int(s3.l)]}_{{{s3.j}}}$ ${s4.n}{orbs[int(s4.l)]}_{{{s4.j}}}$'
                                    
                                    
                                    targets.append(target)
                                    resonants.append(resonant)
                                    min_defects.append(round(float(min_defect), 4))
                                    c_3ks.append(round(float(c3k), 4))
                                    df['n1_0'].append(c['n1_0'])
                                    df['l1_0'].append(c['l1_0'])
                                    df['j1_0'].append(c['j1_0'])
                                    df['n2_0'].append(c['n2_0'])
                                    df['l2_0'].append(c['l2_0'])
                                    df['j2_0'].append(c['j2_0'])
                                    df['n1_1'].append(c['n1_1'])
                                    df['l1_1'].append(c['l1_1'])
                                    df['j1_1'].append(c['j1_1'])
                                    df['n2_1'].append(c['n2_1'])
                                    df['l2_1'].append(c['l2_1'])
                                    df['j2_1'].append(c['j2_1'])
                                    df['defect'].append(c['defect']*1e3)
                                    df['c3k'].append(c['c3k'])

                                    channels.append(c)
                            
                
                    
                    axs[list(changes).index(change2)].set_title(fr'|(n + {change1})${orbs[l1_1]}_{{j_\alpha}}$, (n + {dif} + {change2})${orbs[l2_1]}_{{j_\beta}}$ $\rangle$')
                    axs[list(changes).index(change2)].set_ylabel(f'defect [{energy_unit}]')
                    axs[list(changes).index(change2)].axhline(0, color='grey')
                    axs[list(changes).index(change2)].set_ylim((-20,20))
                    axs[-1].set_xlabel('n')

                    for i, data in enumerate(jpdelta):

                        axs[list(changes).index(change2)].scatter(ns, data, marker='.', label = fr'$j_\alpha$, $j_\beta$ = {labels[i]}', color=colours[i])
                        
                        

                axs[0].legend()
                # fig.suptitle(fr'|{atom1} n${orbs[l1]}_{{{j1}}}$, {atom2} (n + {dif})${orbs[l2]}_{{{j2}}}$ $\rangle$')
                fig.suptitle(fr'|{atom1} n${orbs[l1]}_{{{j1}}}$, {atom2} (n + {dif})${orbs[l2]}_{{{j2}}}$ $\rangle$ $\rightarrow$' + '\n' +
                              fr'|{atom1} [n + {change1}]${orbs[l1_1]}_{{j_p}}$, {atom2} [(n + {dif}) + (n change 2)]${orbs[l2_1]}_{{j_p}}$ $\rangle$')
                # fig.subplots_adjust(top=0.8, left=.1, right=2.7)
                fig.tight_layout()
                path = fr'C:\\Users\\tommy\\OneDrive - Durham University\\level 4 Project\\Figures\\defects\\{atom1}{atom2}\\{orbs[l1]}{orbs[l2]}\\to {orbs[l1_1]}{orbs[l2_1]}\\j1j2({j1},{j2})'
                
                if not os.path.exists(path):
                    os.makedirs(path)
                plt.savefig(path + f'\\dif={dif}, change1={change1}.png')
                # plt.show()
                print(f'dif {dif} change1 {change1}')
                plt.close()
            # print(change1, change2)   

The single-channel quantum defect theory can be inaccurate for effective principal quantum numbers < 25. This can lead to inaccurate matrix elements.


dif -8 change1 -3
1
channel = StateAtom(1.00 |Rb:83,P_3/2,3/2⟩), StateAtom(1.00 |Rb:75,P_3/2,3/2⟩) -> StateAtom(1.00 |Rb:81,D_5/2,5/2⟩), StateAtom(1.00 |Rb:74,S_1/2,1/2⟩)
min defect = 47.22222569398582 MHz
c3k = 1.7533748586080482
2
channel = StateAtom(1.00 |Rb:84,P_3/2,3/2⟩), StateAtom(1.00 |Rb:76,P_3/2,3/2⟩) -> StateAtom(1.00 |Rb:82,D_5/2,5/2⟩), StateAtom(1.00 |Rb:75,S_1/2,1/2⟩)
min defect = 7.910698768682778 MHz
c3k = 1.849198888755288
3
channel = StateAtom(1.00 |Rb:85,P_3/2,3/2⟩), StateAtom(1.00 |Rb:77,P_3/2,3/2⟩) -> StateAtom(1.00 |Rb:83,D_5/2,5/2⟩), StateAtom(1.00 |Rb:76,S_1/2,1/2⟩)
min defect = -27.51806704327464 MHz
c3k = 1.9488930437951055
dif -8 change1 -2
dif -8 change1 -1
dif -8 change1 0
dif -8 change1 1
dif -8 change1 2
dif -8 change1 3
dif -7 change1 -3
4
channel = StateAtom(1.00 |Rb:74,P_3/2,3/2⟩), StateAtom(1.00 |Rb:67,P_3/2,3/2⟩) -> StateAtom(1.00 |Rb:72,D_5/2,5/2⟩), StateAtom(1.00 |Rb:66,S_1/2,1/2⟩)
min defect = 28.477506013587117 MHz
c3k = 1.083198602434205
5
channe

In [4]:
np.set_printoptions(legacy='1.25')
print(len(channels)) 
print(df)


5
{'n1_0': [83, 84, 85, 74, 75], 'l1_0': [1, 1, 1, 1, 1], 'j1_0': [1.5, 1.5, 1.5, 1.5, 1.5], 'n2_0': [75, 76, 77, 67, 68], 'l2_0': [1, 1, 1, 1, 1], 'j2_0': [1.5, 1.5, 1.5, 1.5, 1.5], 'n1_1': [81, 82, 83, 72, 73], 'l1_1': [2, 2, 2, 2, 2], 'j1_1': [2.5, 2.5, 2.5, 2.5, 2.5], 'n2_1': [74, 75, 76, 66, 67], 'l2_1': [0, 0, 0, 0, 0], 'j2_1': [0.5, 0.5, 0.5, 0.5, 0.5], 'defect': [47.22222569398582, 7.910698768682778, -27.51806704327464, 28.477506013587117, -32.123781624250114], 'c3k': [1.7533748586080482, 1.849198888755288, 1.9488930437951055, 1.083198602434205, 1.1501451883922962]}


In [5]:

difs = [na - nb for na, nb in zip([c['n1_0'] for c in channels], [c['n2_0'] for c in channels])]
changes = [((c['n1_1'] - c['n1_0']), (c['n2_1'] - c['n2_0'])) for c in channels]
df.update({'na-nb' : difs, 'n changes (a,b)' : changes})
df = pd.DataFrame(df)
df = df.sort_values(by='n1_0', ascending=True)



In [54]:
from csv import writer
from openpyxl import load_workbook
import xlsxwriter
path = fr'C:\\Users\\tommy\\OneDrive - Durham University\\level 4 Project\\Data\\spreadsheets\\{atom1}{atom2}\\{orbs[l1]}{orbs[l2]} to {orbs[l1_1]}{orbs[l2_1]} j1j2({j1},{j2}).xlsx'


if os.path.exists(path):
    print(f'delete {atom1}{atom2} {orbs[l1]}{orbs[l2]} to {orbs[l1_1]}{orbs[l2_1]} j1j2({j1},{j2}) first')


else:
   df.to_excel(path, index=False)

print(df)

   n1_0  l1_0  j1_0  n2_0  l2_0  j2_0  n1_1  l1_1  j1_1  n2_1  l2_1  j2_1  \
2    44     1   1.5    52     1   0.5    43     2   2.5    52     0   0.5   
0    45     1   1.5    53     1   0.5    44     2   1.5    53     0   0.5   
3    45     1   1.5    53     1   0.5    44     2   2.5    53     0   0.5   
1    46     1   1.5    54     1   0.5    45     2   1.5    54     0   0.5   
4    46     1   1.5    54     1   0.5    45     2   2.5    54     0   0.5   

       defect       c3k  na-nb n changes (a,b)  
2  282.744412  2.287282     -8         (-1, 0)  
0 -123.492658  1.021062     -8         (-1, 0)  
3   14.404369  2.498482     -8         (-1, 0)  
1 -339.418256  1.113207     -8         (-1, 0)  
4 -210.759876  2.723949     -8         (-1, 0)  


In [ ]:
path = f'channel_data\\{atom1}{atom2}\\{orbs[l1]}_{j1}{orbs[l2]}_{j2}'
if not os.path.exists(path):
    os.makedirs(path)

df = {'Target States' : targets, 'Resonant States' : resonants, 'Defect (MHz)' : min_defects, 'C3k (GHz $\mu m^3$)' : c_3ks, '$n_a$ - $n_b$' : difs, 'Changes ($n_a$, $n_b$)' : changes}
df = pd.DataFrame(df)
print(df['Changes (n_a, n_b)'])


In [ ]:
data_df = {'n1_0' : [c['n1_0'] for c in channels],
            'l1_0' : [c['l1_0'] for c in channels],
            'j1_0' : [c['j1_0'] for c in channels],
            'n2_0' : [c['n2_0'] for c in channels],
            'l2_0' : [c['l2_0'] for c in channels],
            'j2_0' : [c['j2_0'] for c in channels],
            'n1_1' : [c['n1_1'] for c in channels],
            'l1_1' : [c['l1_1'] for c in channels],
            'j1_1' : [c['j1_1'] for c in channels],
            'n2_1' : [c['n2_1'] for c in channels],
            'l2_1' : [c['l2_1'] for c in channels],
            'j2_1' : [c['j2_1'] for c in channels],
            'Defect (GHz)' : [c['defect'] for c in channels],
            'C3k (GHz um^3)' : [c['c3k'] for c in channels]} 
data_df= pd.DataFrame(data_df)
data_df.to_excel(f'{path}\\RbCs', sheet_name='{orbs[l1]}_{j1} {orbs[l2]}_{j2}-{orbs[l1_1]} {orbs[l2_1]}.xlsx', index=False)
# # print(df['Changes (n_a, n_b)'])
# print(df.to_latex(index=False,
#             header=True,
#             float_format="%.2f"
#             ))

In [ ]:
n1s = [c['n1_0'] for c in channels]
del_n = [abs(c['n1_0']-c['n2_0']) for c in channels]
defects = [abs(1/c['defect']) for c in channels]
c3ks = [c['c3k'] for c in channels]
print(del_n)
fig, ax = plt.subplots() 
# plt.scatter(n1s, c3ks) 
scat = ax.scatter(
        n1s,
        c3ks,
        c=del_n,
        cmap='Dark2_r',
        marker = '.',
        alpha = defects/np.ones_like(defects) / max(defects, key=abs),
        linewidths = c3ks/np.linalg.norm(c3ks)
    )
ax.set_yscale('log')
print(np.arange(min(del_n, key=abs), max(del_n)-.5))
print((np.array(del_n)+.5*np.ones(len(del_n)))[0:-1])
fig.colorbar(scat, ax=ax, label=r"|$\Delta$ n|", boundaries=np.arange(0.5, max(del_n)+1.5,1), ticks = np.arange(min(del_n, key=abs), max(del_n)+1))
plt.ylabel(r'$C_{{3,k}}$ (GHz $\mu m^3$)')
plt.xlabel('n1')

fig.savefig(path + f'\\C3k_vs_n1.png')


In [ ]:
tunes_list = np.zeros(len(channels))
E_spacing = np.zeros_like(channels)
df.update({'Electric Field Tunes (V/cm)' : tunes_list})
df.update({'Electric Field Spacing (V/cm)' : E_spacing})
df=DataFrame(df, index=np.arange(1, len(targets)+1))
df_excel = df
print(df)



In [ ]:
from pairinteraction.visualization.colormaps import alphamagma
import os

maxe = .15
mine=0
num=500
electric_fields = np.linspace(mine,maxe,num)
n_dif = 3
jdif = 3
ldif = 3
b=0
energy_unit = 'MHz'

channel=22
c = channels[channel-1]

l1, l2 = c['l1_0'], c['l2_0']
l1_1, l2_1 = c['l1_1'], c['l2_1']

j1, j2 = c['j1_0'], c['j2_0']
j1_1, j2_1 = c['j1_1'], c['j2_1']



newpath = fr'C:\\Users\\tommy_ybkgmpp\\OneDrive - Durham University\\level 4 Project\\Figures\\Stark Plots\\{atom1}{atom2}\\{orbs[l1]}{orbs[l2]}\\to {orbs[l1_1]}{orbs[l2_1]}\\j1j2({j1},{j2})\\'
if not os.path.exists(newpath):
    os.makedirs(newpath)

n1=c['n1_0']
n2=c['n2_0']
ket1 = pi.KetAtom(atom1, n=n1, l=l1, m=j1, j=j1)
ket1basis = pi.BasisAtom(atom1, n=(ket1.n,ket1.n), l=(ket1.l,ket1.l), j=(ket1.j,ket1.j), m=(ket1.m,ket1.m))
# print(type(ket1basis.states))
ket2 = pi.KetAtom(atom2, n=n2, l=l2, m=j2, j=j2)
ket2basis = pi.BasisAtom(atom2, n=(ket2.n,ket2.n), l=(ket2.l,ket2.l), j=(ket2.j,ket2.j))

ket1_energy = ket1.get_energy(unit=energy_unit)
# print('ket1 energy = ' + str(ket1_energy) + energy_unit)
basis1 = pi.BasisAtom(atom1, n=(ket1.n-n_dif, ket1.n+n_dif), l=(0,ket1.l+ldif), j=(1/2,ket1.j+jdif), m=(-ket1.j-jdif, ket1.j+jdif))

ket1systems = [pi.SystemAtom(ket1basis).set_magnetic_field([0, 0, b], unit='G').set_electric_field([0, 0, e], unit="V/cm") for e in electric_fields
]
pi.diagonalize(ket1systems, diagonalizer="eigen", sort_by_energy=True)
ket1energies = [system.get_eigenenergies(unit=energy_unit) - ket1_energy for system in ket1systems]

systems = [
    pi.SystemAtom(basis1).set_magnetic_field([0, 0, b], unit='G').set_electric_field([0, 0, e], unit="V/cm") for e in electric_fields
]
pi.diagonalize(systems, diagonalizer="eigen", sort_by_energy=True)
eigenenergies1 = [system.get_eigenenergies(unit=energy_unit) - ket1_energy for system in systems]
overlaps1 = [system.get_eigenbasis().get_overlaps(ket1) for system in systems]

ket2_energy = ket2.get_energy(unit=energy_unit)
# print('ket2 energy = ' + str(ket2_energy) + energy_unit)

basis2 = pi.BasisAtom(atom2, n=(ket2.n-n_dif, ket2.n+n_dif), l=(0,ket2.l+2), j=(1/2,ket2.j+1), m=(-ket2.j-1,ket2.j+1))

ket2systems = [pi.SystemAtom(ket2basis).set_magnetic_field([0, 0, b], unit='G').set_electric_field([0, 0, e], unit="V/cm") for e in electric_fields
]
pi.diagonalize(ket2systems, diagonalizer="eigen", sort_by_energy=True)
ket2energies = [system.get_eigenenergies(unit=energy_unit) - ket2_energy for system in ket2systems]

systems = [
    pi.SystemAtom(basis2).set_magnetic_field([0, 0, b], unit='G').set_electric_field([0, 0, -e], unit="V/cm") for e in electric_fields
]
pi.diagonalize(systems, diagonalizer="eigen", sort_by_energy=True)
eigenenergies2 = [system.get_eigenenergies(unit=energy_unit) - ket2_energy for system in systems]
overlaps2 = [system.get_eigenbasis().get_overlaps(ket2) for system in systems]

# Finding defect

ket1_1 = pi.KetAtom(atom1, n=c['n1_1'], l=l1_1, j=j1_1, m=j1_1)

ket2_1 = pi.KetAtom(atom2, n=c['n2_1'], l=l2_1, j=j2_1, m=j2_1)
defect = ket1_1.get_energy(energy_unit) + ket2_1.get_energy(energy_unit) - ket1.get_energy(energy_unit) - ket2.get_energy(energy_unit)
    

In [ ]:
fig, ax = plt.subplots(2,1, sharex=True, figsize=(5,8))
ylim = 100
ax[0].set_ylabel(f"Energy [{energy_unit}]")

ax[0].plot(electric_fields, np.array(eigenenergies1), c=".1", lw=0.5)

ax[0].set_title(str(ket1))
ax[0].set_ylim(-ylim,ylim)
x_repeated = np.hstack(
    [val * np.ones_like(es) for val, es in zip(electric_fields, eigenenergies1)]
)

energies_flattened1 = np.hstack(eigenenergies1)
overlaps_flattened1 = np.hstack(overlaps1)
sorter1 = np.argsort(overlaps_flattened1)

scat = ax[0].scatter(
    x_repeated[sorter1],
    energies_flattened1[sorter1],
    c=overlaps_flattened1[sorter1],
    s=15,
    vmin=0,
    vmax=1,
    cmap=alphamagma,
    marker='.'
)
fig.colorbar(scat, ax=ax, label="Overlap with state of interest", aspect=30, shrink=.5)

ax[1].set_ylabel(f"Energy [{energy_unit}]")

ax[1].plot(electric_fields, np.array(eigenenergies2), c=".1", lw=0.5)
ax[1].set_title(str(ket2))
ax[1].set_ylim(-ylim,ylim)
x_repeated = np.hstack(
    [val * np.ones_like(es) for val, es in zip(electric_fields, eigenenergies2)]
)

energies_flattened2 = np.hstack(eigenenergies2)
overlaps_flattened2 = np.hstack(overlaps2)
sorter2 = np.argsort(overlaps_flattened2)

scat = ax[1].scatter(
    x_repeated[sorter2],
    energies_flattened2[sorter2],
    c=overlaps_flattened2[sorter2],
    s=15,
    vmin=0,
    vmax=1,
    cmap=alphamagma,
    marker='.'
)
plt.xlabel("Electric Field [V/cm]")
plt.suptitle(fr'basis: n$\pm${n_dif}, j = (1/2, j + {jdif}), m = (-(j+{jdif}), j+{jdif})' '\n' f'l=(0, {ldif}), B = {b}G')

fig = plt.figure(figsize = (8,5))
energy_lim = abs(defect + defect/3)
eigenenergies_pair = []
state0s = []

for i, channel1 in enumerate(np.transpose(eigenenergies1)):
    for j, channel2 in enumerate(np.transpose(eigenenergies2)):
        eigenenergy = channel1 + channel2
        if abs(channel1[0]) <= abs(ket1_1.get_energy(energy_unit)-ket1_energy)+2 and abs(channel1[0]) <= abs(ket1_1.get_energy(energy_unit)-ket1_energy)+2:
            plt.plot(electric_fields, eigenenergy, c=".1", lw=0.25)
            

            eigenenergies_pair.append(eigenenergy)
            if  eigenenergy[0] == 0:
                state0s.append(eigenenergy)
                # print(state0[-1])
offset=defect/2
min_energy = -energy_lim+offset
max_energy = energy_lim+offset
plt.title(f'{str(ket1)}{str(ket2)}')
plt.ylim(min_energy, max_energy)
plt.ylabel(f"Energy [{energy_unit}]")
plt.xlabel("Electric Field [V/cm]")

tunes=[]
for eigenenergy in eigenenergies_pair:
    if eigenenergy[0] == state0s[0][0]:
        continue
    elif round(defect, 3)-.5 <= round(eigenenergy[0],2) <= round(defect, 3)+.5:

        for state0 in state0s:
            
            energy = eigenenergy-state0
            cross = min(energy, key=abs)
            if abs(cross) < 1:
                index = list(energy).index(cross)
                if index == 0 or index == len(electric_fields)-1:
                    continue
                else:
                    tuned = electric_fields[index]
                    plt.vlines(x=tuned, ymin = min_energy, ymax = eigenenergy[index], linestyles='dashed', colors='magenta', linewidth = 0.5)
                    tunes.append(format(round(tuned, 4), ".15g"))

plt.axhline(defect, color='r', linewidth=2, alpha=1, linestyle=(0, (1, 10)))
print(set(tunes))
tunes = set(tunes)


In [ ]:

print(len(tunes))
if len(tunes) == 0:
    tunes = f'None found between {mine} and {maxe}'
df.at[channel, 'Electric Field Tunes (V/cm)'] = str(tunes)
df.at[channel, 'Electric Field Spacing (V/cm)'] = (mine-maxe)/num
print(df['Electric Field Tunes (V/cm)'])
print(df['Electric Field Spacing (V/cm)'])



In [ ]:

df.to_excel(f'{path}\\{orbs[l1]}_{j1} {orbs[l2]}_{j2}-{orbs[l1_1]} {orbs[l2_1]}.xlsx', index=True)

print(df.to_latex(index=True,
            header=True,
            ))

In [ ]:
fig, ax = plt.subplots(2,1, sharex=True, figsize=(5,8))
ylim = 100
ax[0].set_ylabel(f"Energy [{energy_unit}]")

ax[0].plot(magnetic_fields, np.array(eigenenergies1), c=".1", lw=0.5)

ax[0].set_title(str(ket1))
ax[0].set_ylim(-ylim,ylim)
x_repeated = np.hstack(
    [val * np.ones_like(es) for val, es in zip(magnetic_fields, eigenenergies1)]
)

energies_flattened1 = np.hstack(eigenenergies1)
overlaps_flattened1 = np.hstack(overlaps1)
sorter1 = np.argsort(overlaps_flattened1)

scat = ax[0].scatter(
    x_repeated[sorter1],
    energies_flattened1[sorter1],
    c=overlaps_flattened1[sorter1],
    s=15,
    vmin=0,
    vmax=1,
    cmap=alphamagma,
    marker='.'
)
fig.colorbar(scat, ax=ax, label="Overlap with state of interest", aspect=30, shrink=.5)

ax[1].set_ylabel(f"Energy [{energy_unit}]")

ax[1].plot(magnetic_fields, np.array(eigenenergies2), c=".1", lw=0.5)
ax[1].set_title(str(ket2))
ax[1].set_ylim(-ylim,ylim)
x_repeated = np.hstack(
    [val * np.ones_like(es) for val, es in zip(magnetic_fields, eigenenergies2)]
)

energies_flattened2 = np.hstack(eigenenergies2)
overlaps_flattened2 = np.hstack(overlaps2)
sorter2 = np.argsort(overlaps_flattened2)

scat = ax[1].scatter(
    x_repeated[sorter2],
    energies_flattened2[sorter2],
    c=overlaps_flattened2[sorter2],
    s=15,
    vmin=0,
    vmax=1,
    cmap=alphamagma,
    marker='.'
)
plt.xlabel("magnetic Field [V/cm]")
plt.suptitle(fr'basis: n$\pm${n_dif}, j = (1/2, j + {jdif}), m = (-(j+{jdif}), j+{jdif})' '\n' f'l=(0, {ldif}), B = {b}G')

fig = plt.figure(figsize = (8,5))
energy_lim = abs(defect + defect/3)
eigenenergies_pair = []
state0s = []

for i, channel1 in enumerate(np.transpose(eigenenergies1)):
    for j, channel2 in enumerate(np.transpose(eigenenergies2)):
        eigenenergy = channel1 + channel2
        if abs(channel1[0]) <= abs(ket1_1.get_energy(energy_unit)-ket1_energy)+2 and abs(channel1[0]) <= abs(ket1_1.get_energy(energy_unit)-ket1_energy)+2:
            plt.plot(magnetic_fields, eigenenergy, c=".1", lw=0.25)
            

            eigenenergies_pair.append(eigenenergy)
            if  eigenenergy[0] == 0:
                state0s.append(eigenenergy)
                # print(state0[-1])
offset=defect/2
min_energy = -energy_lim+offset
max_energy = energy_lim+offset
plt.title(f'{str(ket1)}{str(ket2)}')
plt.ylim(min_energy, max_energy)
plt.ylabel(f"Energy [{energy_unit}]")
plt.xlabel("Magnetic Field [G]")

tunes=[]
for eigenenergy in eigenenergies_pair:
    if eigenenergy[0] == state0s[0][0]:
        continue
    elif round(defect, 3)-.5 <= round(eigenenergy[0],2) <= round(defect, 3)+.5:

        for state0 in state0s:
            
            energy = eigenenergy-state0
            cross = min(energy, key=abs)
            if abs(cross) < 1:
                index = list(energy).index(cross)
                if index == 0 or index == len(magnetic_fields)-1:
                    continue
                else:
                    tuned = magnetic_fields[index]
                    plt.vlines(x=tuned, ymin = min_energy, ymax = eigenenergy[index], linestyles='dashed', colors='magenta', linewidth = 0.5)
                    tunes.append(format(round(tuned, 4), ".15g"))

plt.axhline(defect, color='r', linewidth=2, alpha=1, linestyle='dotted')
print(set(tunes))
tunes = set(tunes)


In [ ]:
fig.savefig(newpath + f'channel {channel}.png')